In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import numpy as np
import editdistance 
from TGNHTR import TGNHTR
from tqdm.auto import tqdm

def to_adj(edge_index, num_nodes):
    a = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    for i, j in edge_index:
        a[i, j] = 1.0
    return a

def make_batch(samples):
    x_list, a_list, time_list = [], [], []
    y_list, in_len, tgt_len = [], [], []

    for s in samples:
        x = s["nodes"].astype(np.float32)
        edge_index = s["edge_index"]

        N = len(x)
        a = np.zeros((N, N), dtype=np.float32)
        for i, j in edge_index:
            a[i, j] = 1.0

        x_list.append(x)
        a_list.append(a)
        time_list.append(np.arange(N))

        y_list.append(s["target"])
        in_len.append(N)
        tgt_len.append(len(s["target"]))

    return x_list, a_list, time_list, y_list, in_len, tgt_len

def compute_ctc_loss(y_true, logits, input_lengths, target_lengths):
    y_pred = tf.nn.softmax(logits, axis=-1)
    loss = tf.keras.backend.ctc_batch_cost(
        y_true, y_pred, input_lengths, target_lengths
    )
    return tf.reduce_mean(loss)

def cer(ref, hyp):
    ref_chars = list(ref)
    hyp_chars = list(hyp)
    if len(ref_chars) == 0:
        return 0.0
    return editdistance.eval(ref_chars, hyp_chars) / len(ref_chars)

def wer(ref, hyp):
    ref_words = ref.split()
    hyp_words = hyp.split()
    if len(ref_words) == 0:
        return 0.0
    return editdistance.eval(ref_words, hyp_words) / len(ref_words)

def decode_batch(logits, id2char):
    lengths = np.full(logits.shape[0], logits.shape[1])
    decoded, _ = tf.keras.backend.ctc_decode(logits, lengths, greedy=True)
    decoded = decoded[0].numpy()
    texts = []
    for row in decoded:
        chars = []
        for idx in row:
            if idx < 0 or idx == 0:
                continue
            chars.append(id2char[idx])
        texts.append("".join(chars))
    return texts

def pad_graph_batch(x_list, a_list, time_list):
    max_nodes = max(x.shape[0] for x in x_list)
    max_seq = max(len(t) for t in time_list)
    x_padded, a_padded, time_padded = [], [], []
    for x, a, t in zip(x_list, a_list, time_list):
        pad_n = max_nodes - x.shape[0]
        x_padded.append(np.pad(x, ((0, pad_n), (0, 0)), 'constant'))
        a_padded.append(np.pad(a, ((0, pad_n), (0, pad_n)), 'constant'))
        pad_s = max_seq - len(t)
        time_padded.append(np.pad(t, (0, pad_s), 'constant', constant_values=0))
    return (
        np.array(x_padded, dtype=np.float32),
        np.array(a_padded, dtype=np.float32),
        np.array(time_padded, dtype=np.int32),
    )

def evaluate(model, dataset, id2char, batch_size=8):
    cer_scores, wer_scores = [], []
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i + batch_size]
        x_list, a_list, time_idx, y_list, input_lengths, target_lengths = make_batch(batch)
        x_batch, a_batch, time_batch = pad_graph_batch(x_list, a_list, time_idx)
        logits = model(
            [tf.constant(x_batch), tf.constant(a_batch), tf.constant(time_batch)],
            training=False
        )
        preds = decode_batch(logits.numpy(), id2char)
        refs = ["".join([id2char.get(idx, "") for idx in y]) for y in y_list]
        for r, p in zip(refs, preds):
            cer_scores.append(cer(r, p))
            wer_scores.append(wer(r, p))
    return np.mean(cer_scores), np.mean(wer_scores)

def load_letters(path):
    letters = []
    with open(path, encoding="utf8") as f:
        for line in f:
            token = line.strip()
            if token:
                letters.append(token)
    return letters

def build_ctc_vocab(letters):
    charset = set(letters)
    charset.add(" ")
    charset.add("'")
    charset = sorted(charset)

    char2id = {"<blank>": 0, "<unk>": 1}
    for idx, ch in enumerate(charset, start=2):
        char2id[ch] = idx

    id2char = {v: k for k, v in char2id.items()}
    return char2id, id2char

letters = load_letters("Technical files/letters")
char2id, id2char = build_ctc_vocab(letters)


len_char2id = len(char2id)  # не хардкодим, берём из реального словаря

optimizer = tf.keras.optimizers.Adam(1e-3)

model = TGNHTR(vocab_size=len_char2id)

import pickle

with open("train.pkl", "rb") as f:
    train = pickle.load(f)

with open("val1.pkl", "rb") as f:
    val_1 = pickle.load(f)

with open("val2.pkl", "rb") as f:
    val_2 = pickle.load(f)

with open("test.pkl", "rb") as f:
    test = pickle.load(f)

print("Прогрев модели (инициализация графов)...")
BATCH_SIZE = 16

dummy_batch = train[0:BATCH_SIZE]
dx_list, da_list, dt_list, _, _, _ = make_batch(dummy_batch)
dx_batch, da_batch, dt_batch = pad_graph_batch(dx_list, da_list, dt_list)

_ = model(
    [tf.constant(dx_batch), tf.constant(da_batch), tf.constant(dt_batch)],
    training=False
)
print("Прогрев завершен! Модель построена.")

@tf.function(reduce_retracing=True)
def train_step(x_batch, a_batch, time_batch, y_padded, in_len, tgt_len):
    with tf.GradientTape() as tape:
        logits = model([x_batch, a_batch, time_batch], training=True)

        loss = tf.nn.ctc_loss(
            labels=y_padded,
            logits=logits,
            label_length=tgt_len,
            logit_length=in_len,
            logits_time_major=False,
            blank_index=-1
        )
        loss = tf.reduce_mean(loss)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

EPOCHS = 5
total_steps = len(train) // BATCH_SIZE
for epoch in range(EPOCHS):
    losses = []
    print(f"\n--- Эпоха {epoch + 1}/{EPOCHS} ---")

    with tqdm(total=total_steps, desc="Обучение") as pbar:
        for i in range(0, len(train) - BATCH_SIZE + 1, BATCH_SIZE):
            batch = train[i:i + BATCH_SIZE]

            x_list, a_list, time_list, y_list, input_length, target_length = make_batch(batch)

            x_batch, a_batch, time_batch = pad_graph_batch(x_list, a_list, time_list)

            in_len = np.array(input_length, dtype=np.int32)
            tgt_len = np.array(target_length, dtype=np.int32)
            y_padded = tf.keras.utils.pad_sequences(y_list, padding='post', value=0)

            loss = train_step(
                tf.constant(x_batch),
                tf.constant(a_batch),
                tf.constant(time_batch),
                tf.constant(y_padded, dtype=tf.int32),
                tf.constant(in_len),
                tf.constant(tgt_len),
            )

            current_loss = float(loss)
            losses.append(current_loss)
            pbar.set_postfix({'loss': f"{current_loss:.4f}"})
            pbar.update(1)

    mean_cer, mean_wer = evaluate(model, val_1, id2char, batch_size=BATCH_SIZE)
    print(f"Val1 — CER: {mean_cer:.4f} | WER: {mean_wer:.4f}")